# SAE Operational Fingerprint — Math Operations on Gemma-3-1B

**Core idea:** Train one shared SAE on a large mixed corpus of math + non-math activations.
The dictionary learns a common basis. Operation fingerprints are then extracted as
the subset of features that activate differentially for each operation.

## Why shared SAE?
- Dictionaries stay aligned — feature 42 means the same thing for addition and multiplication
- Directly testable: is `fingerprint(add) ⊂ fingerprint(mul)`?
- Non-math activations act as a baseline, forcing operational features to be distinctive

## Operations
| Label | Template | Example |
|-------|----------|---------|
| `add` | `{a}+{b}=` | `12+34=` |
| `mul` | `{a}*{b}=` | `6*7=` |
| `sub` | `{a}-{b}=` | `45-13=` |
| `ctrl` | misc non-math | `The capital of France is` |

## Pipeline
```
1. Generate prompts     → math (add/mul/sub) + non-math control
2. Measure accuracy     → keep track of which the model actually solves
3. Collect activations  → Gemma-3-1B residual stream, layer 17, last token
4. Train shared SAE     → TopK SAE on ALL activations together
5. Extract fingerprints → per-operation feature sets (effect size threshold)
6. Hierarchy test       → containment: fingerprint(add) ⊂ fingerprint(mul)?
7. RSA                  → representational geometry comparison
8. Constant-trace       → variance within operation predicts accuracy
```

⚠️ Gemma is gated. Accept terms at https://huggingface.co/google/gemma-3-1b-pt and set `HF_TOKEN`.

In [ ]:
!pip install -q transformer_lens

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
HF_TOKEN   = "hf_..."         # your HuggingFace token
MODEL_NAME = "google/gemma-3-1b-pt"
LAYER      = 17               # residual stream; Gemma-3-1B has 26 layers

# Data — start with 2000/op to validate, scale to 5000+ once it works
N_PER_OP   = 2000             # math examples per operation (add / mul / sub)
N_CTRL     = 1000             # non-math control examples

# Operand ranges to sample from
RANGES = {
    "single": (1, 9),         # single-digit  — model almost always correct
    "double": (10, 99),       # two-digit     — more variable
}

# SAE
SAE_K      = 32               # TopK sparsity: active features per token
SAE_RATIO  = 8                # d_sae = d_model × SAE_RATIO
SAE_LR     = 3e-4
SAE_EPOCHS = 15
SAE_BATCH  = 512

# Fingerprint threshold: features with Cohen's d > FP_THRESHOLD vs ctrl
FP_THRESHOLD = 0.5            # medium effect size

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import random, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from huggingface_hub import login
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
from numpy.linalg import norm
from transformer_lens import HookedTransformer
from tqdm.auto import tqdm

login(token=HF_TOKEN, add_to_git_credential=False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
model = HookedTransformer.from_pretrained(
    MODEL_NAME, center_writing_weights=False, dtype="bfloat16"
)
model.eval().to(DEVICE)
D_MODEL = model.cfg.d_model
D_SAE   = D_MODEL * SAE_RATIO
HOOK    = f"blocks.{LAYER}.hook_resid_pre"
print(f"d_model={D_MODEL}  d_sae={D_SAE}  hook={HOOK}")

In [ ]:
# ── 1. Data generation ────────────────────────────────────────────────────────
rng = random.Random(0)

def sample_pair(lo, hi):
    return rng.randint(lo, hi), rng.randint(lo, hi)

def make_math_data(op_label, n):
    """Generate n math prompts balanced across single/double digit ranges."""
    data = []
    per_range = n // len(RANGES)
    for range_name, (lo, hi) in RANGES.items():
        for _ in range(per_range):
            a, b = sample_pair(lo, hi)
            if op_label == "add":
                prompt, expected = f"{a}+{b}=", a + b
            elif op_label == "mul":
                prompt, expected = f"{a}*{b}=", a * b
            elif op_label == "sub":
                # keep result non-negative for cleaner token prediction
                a, b = max(a, b), min(a, b)
                prompt, expected = f"{a}-{b}=", a - b
            data.append(dict(prompt=prompt, op=op_label, range=range_name,
                             a=a, b=b, expected=expected))
    return data

# Non-math control prompts — diverse surface forms
CTRL_TEMPLATES = [
    "The capital of {country} is",
    "The color of the sky is",
    "She walked into the {room} and",
    "The largest {animal} in the world is",
    "In {year}, the {event}",
    "He picked up the {obj} and",
    "The temperature today is",
    "Scientists discovered that",
]
CTRL_FILLS = dict(
    country=["France","Germany","Japan","Brazil","Canada","Italy","Spain","India"],
    room=["kitchen","library","office","garden","basement","hall"],
    animal=["mammal","reptile","bird","fish","insect"],
    year=["1900","1950","1980","2001","2010","2023"],
    event=["war ended","discovery was made","team won","law passed"],
    obj=["book","pen","phone","key","bag","cup"],
)

def make_ctrl_data(n):
    data = []
    for _ in range(n):
        tmpl = rng.choice(CTRL_TEMPLATES)
        filled = tmpl
        for k, opts in CTRL_FILLS.items():
            if f"{{{k}}}" in filled:
                filled = filled.replace(f"{{{k}}}", rng.choice(opts))
        data.append(dict(prompt=filled, op="ctrl", range="none", a=None, b=None, expected=None))
    return data

add_data  = make_math_data("add", N_PER_OP)
mul_data  = make_math_data("mul", N_PER_OP)
sub_data  = make_math_data("sub", N_PER_OP)
ctrl_data = make_ctrl_data(N_CTRL)
all_data  = add_data + mul_data + sub_data + ctrl_data
rng.shuffle(all_data)

print(f"add: {len(add_data)}  mul: {len(mul_data)}  sub: {len(sub_data)}  ctrl: {len(ctrl_data)}")
print("Samples:")
for d in [add_data[0], mul_data[0], sub_data[0], ctrl_data[0]]:
    print(f"  [{d['op']:4s}] {d['prompt']!r:30s}  expected={d['expected']}")

In [ ]:
# ── 2. Accuracy measurement ───────────────────────────────────────────────────
# We measure accuracy before activation collection so we can stratify results
# by correct/incorrect. Single-token greedy decode — works for answers < ~1000.

@torch.no_grad()
def measure_accuracy(records, batch_size=16):
    results = []
    for i in range(0, len(records), batch_size):
        batch  = records[i:i+batch_size]
        tokens = model.to_tokens([r["prompt"] for r in batch], prepend_bos=True)
        logits = model(tokens)[:, -1, :].float()
        for j, rec in enumerate(batch):
            if rec["expected"] is None:
                results.append(None)
                continue
            pred = model.tokenizer.decode([logits[j].argmax().item()]).strip()
            results.append(pred == str(rec["expected"]))
    return results

print("Measuring accuracy...")
for op, data in [("add", add_data), ("mul", mul_data), ("sub", sub_data)]:
    correct = measure_accuracy(data)
    for rec, c in zip(data, correct):
        rec["correct"] = c
    for range_name in RANGES:
        grp = [r for r in data if r["range"] == range_name]
        acc = np.mean([r["correct"] for r in grp])
        print(f"  {op:3s} [{range_name:6s}]: {acc:.1%}  (n={len(grp)})")

for rec in ctrl_data:
    rec["correct"] = None

In [ ]:
# ── 3. Collect activations ────────────────────────────────────────────────────
# Residual stream at last token position — this is where the model "decides" the answer.

@torch.no_grad()
def collect_acts(records, batch_size=16):
    all_acts = []
    for i in tqdm(range(0, len(records), batch_size), desc="collecting"):
        batch  = records[i:i+batch_size]
        tokens = model.to_tokens([r["prompt"] for r in batch], prepend_bos=True)
        _, cache = model.run_with_cache(tokens, names_filter=HOOK)
        acts = cache[HOOK][:, -1, :].to(torch.float32).cpu()
        all_acts.append(acts)
    return torch.cat(all_acts, dim=0)

add_acts  = collect_acts(add_data)
mul_acts  = collect_acts(mul_data)
sub_acts  = collect_acts(sub_data)
ctrl_acts = collect_acts(ctrl_data)

# Full corpus for SAE training
train_acts = torch.cat([add_acts, mul_acts, sub_acts, ctrl_acts], dim=0)
print(f"\nTraining corpus: {train_acts.shape}")
print(f"  mean={train_acts.mean():.4f}  std={train_acts.std():.4f}")

In [ ]:
# ── 4. Shared TopK SAE ────────────────────────────────────────────────────────
# One dictionary for all operations. Fingerprints will be defined as
# which features in this shared dictionary are differentially active per operation.

class TopKSAE(nn.Module):
    def __init__(self, d_in, d_sae, k):
        super().__init__()
        self.k = k
        self.pre_bias = nn.Parameter(torch.zeros(d_in))
        self.W_enc    = nn.Parameter(torch.empty(d_in, d_sae))
        self.b_enc    = nn.Parameter(torch.zeros(d_sae))
        self.W_dec    = nn.Parameter(torch.empty(d_sae, d_in))
        self.b_dec    = nn.Parameter(torch.zeros(d_in))
        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self.W_dec.data = nn.functional.normalize(self.W_dec.data, dim=1)

    def encode(self, x):
        pre = (x - self.pre_bias) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        acts = torch.zeros_like(pre)
        acts.scatter_(-1, idx, vals.clamp(min=0))
        return acts

    def decode(self, acts):
        return acts @ self.W_dec + self.b_dec

    def forward(self, x):
        acts  = self.encode(x)
        x_hat = self.decode(acts)
        return acts, x_hat

    @torch.no_grad()
    def normalise_decoder(self):
        self.W_dec.data = nn.functional.normalize(self.W_dec.data, dim=1)


sae = TopKSAE(D_MODEL, D_SAE, SAE_K).to(DEVICE)
print(f"SAE params: {sum(p.numel() for p in sae.parameters()):,}")

In [ ]:
# ── 5. Train SAE ──────────────────────────────────────────────────────────────
AUX_W     = 1/32
DEAD_THR  = 1e-4
optimizer = torch.optim.Adam(sae.parameters(), lr=SAE_LR)
loader    = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(train_acts),
    batch_size=SAE_BATCH, shuffle=True
)
feat_usage = torch.zeros(D_SAE, device=DEVICE)
history    = []

for epoch in range(SAE_EPOCHS):
    e_mse = e_aux = 0.0
    for (x,) in loader:
        x = x.to(DEVICE)
        acts, x_hat = sae(x)
        with torch.no_grad():
            feat_usage = 0.99*feat_usage + 0.01*(acts > 0).float().mean(0)
        mse  = (x - x_hat).pow(2).mean()
        dead = (feat_usage < DEAD_THR).float()
        aux  = ((x - x_hat).detach() - (acts*dead) @ sae.W_dec).pow(2).mean() if dead.sum() > 0 else torch.tensor(0., device=DEVICE)
        loss = mse + AUX_W * aux
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
        optimizer.step(); sae.normalise_decoder()
        e_mse += mse.item(); e_aux += aux.item()
    n = len(loader)
    n_dead = (feat_usage < DEAD_THR).sum().item()
    history.append(dict(epoch=epoch+1, mse=e_mse/n, aux=e_aux/n, dead=n_dead))
    print(f"Epoch {epoch+1:2d}/{SAE_EPOCHS}  mse={e_mse/n:.5f}  dead={n_dead}/{D_SAE}")

# Variance explained
sae.eval()
with torch.no_grad():
    s  = train_acts[:1000].to(DEVICE)
    _, r = sae(s)
    var_expl = 1 - (s-r).pow(2).sum() / s.pow(2).sum()
print(f"\nVariance explained: {var_expl.item():.2%}")
print(f"Live features: {(feat_usage > DEAD_THR).sum().item()}/{D_SAE}")

In [ ]:
# ── 6. Extract SAE features per operation ─────────────────────────────────────
@torch.no_grad()
def to_features(acts_tensor):
    out = []
    for i in range(0, len(acts_tensor), 512):
        out.append(sae.encode(acts_tensor[i:i+512].to(DEVICE)).cpu().numpy())
    return np.concatenate(out, axis=0)

feats = {
    "add":  to_features(add_acts),
    "mul":  to_features(mul_acts),
    "sub":  to_features(sub_acts),
    "ctrl": to_features(ctrl_acts),
}

for op, f in feats.items():
    sparsity = (f == 0).mean()
    print(f"  {op:4s}: shape={f.shape}  sparsity={sparsity:.2%}")

In [ ]:
# ── 7. Fingerprint extraction — Cohen's d vs control ─────────────────────────
# For each feature and each math operation, compute Cohen's d relative to ctrl.
# Fingerprint = features where |d| > FP_THRESHOLD (medium effect size).
# We use one-sided d: features MORE active in the operation than in ctrl.

ctrl_mean = feats["ctrl"].mean(axis=0)
ctrl_std  = feats["ctrl"].std(axis=0) + 1e-8

fingerprints = {}   # op → boolean mask over D_SAE features
effect_sizes = {}   # op → Cohen's d per feature

for op in ["add", "mul", "sub"]:
    op_mean = feats[op].mean(axis=0)
    # Pooled std
    pooled  = np.sqrt((ctrl_std**2 + feats[op].std(axis=0)**2 + 1e-8) / 2)
    d       = (op_mean - ctrl_mean) / pooled   # (D_SAE,)
    effect_sizes[op] = d
    fingerprints[op] = d > FP_THRESHOLD        # features reliably MORE active in op
    n_fp = fingerprints[op].sum()
    print(f"  {op}: {n_fp} fingerprint features  (d > {FP_THRESHOLD})")

# Also compute fingerprint for ctrl (features more active in ctrl than math)
math_mean = np.concatenate([feats["add"], feats["mul"], feats["sub"]], axis=0).mean(axis=0)
effect_sizes["ctrl"] = (ctrl_mean - math_mean) / (ctrl_std + 1e-8)
fingerprints["ctrl"] = effect_sizes["ctrl"] > FP_THRESHOLD

In [ ]:
# ── 8. Hierarchy test: containment ────────────────────────────────────────────
# Core hypothesis: addition's fingerprint features are a subset of multiplication's.
# Metric: containment = |fp(A) ∩ fp(B)| / |fp(A)|  (asymmetric — A contained in B)
# Also report Jaccard for symmetric comparison.

def containment(fp_a, fp_b):
    """Fraction of A's fingerprint features that are also in B's."""
    inter = (fp_a & fp_b).sum()
    return float(inter) / (fp_a.sum() + 1e-8)

def jaccard(fp_a, fp_b):
    inter = (fp_a & fp_b).sum()
    union = (fp_a | fp_b).sum()
    return float(inter) / (union + 1e-8)

ops = ["add", "mul", "sub"]
print("Containment matrix  [row contained-in col]:")
print(f"{'':6s}", "  ".join(f"{o:>6s}" for o in ops))
for a in ops:
    row = [containment(fingerprints[a], fingerprints[b]) for b in ops]
    print(f"{a:6s}", "  ".join(f"{v:>6.3f}" for v in row))

print("\nJaccard similarity matrix:")
print(f"{'':6s}", "  ".join(f"{o:>6s}" for o in ops))
for a in ops:
    row = [jaccard(fingerprints[a], fingerprints[b]) for b in ops]
    print(f"{a:6s}", "  ".join(f"{v:>6.3f}" for v in row))

print("\nHierarchy hypothesis:  containment(add→mul) > containment(mul→add)")
c_am = containment(fingerprints["add"], fingerprints["mul"])
c_ma = containment(fingerprints["mul"], fingerprints["add"])
print(f"  add→mul: {c_am:.3f}   mul→add: {c_ma:.3f}")
verdict = "SUPPORTED" if c_am > c_ma else "NOT SUPPORTED"
print(f"  Verdict: {verdict}")

In [ ]:
# ── 9. Fingerprint overlap visualisation ──────────────────────────────────────
# Venn-style: feature counts in each set / intersection

fp_add = fingerprints["add"]
fp_mul = fingerprints["mul"]
fp_sub = fingerprints["sub"]

only_add = (fp_add & ~fp_mul & ~fp_sub).sum()
only_mul = (~fp_add & fp_mul & ~fp_sub).sum()
only_sub = (~fp_add & ~fp_mul & fp_sub).sum()
add_mul  = (fp_add & fp_mul & ~fp_sub).sum()
add_sub  = (fp_add & ~fp_mul & fp_sub).sum()
mul_sub  = (~fp_add & fp_mul & fp_sub).sum()
all_ops  = (fp_add & fp_mul & fp_sub).sum()

print("Feature overlap (# fingerprint features):")
print(f"  Only add:          {only_add}")
print(f"  Only mul:          {only_mul}")
print(f"  Only sub:          {only_sub}")
print(f"  add ∩ mul:         {add_mul}")
print(f"  add ∩ sub:         {add_sub}")
print(f"  mul ∩ sub:         {mul_sub}")
print(f"  add ∩ mul ∩ sub:   {all_ops}  ← shared math core")

# Effect size distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {"add": "steelblue", "mul": "darkorange", "sub": "green"}
for ax, op in zip(axes, ["add","mul","sub"]):
    d = effect_sizes[op]
    ax.hist(d[d > 0], bins=60, color=colors[op], alpha=0.7, label=op)
    ax.axvline(FP_THRESHOLD, color="red", ls="--", label=f"threshold={FP_THRESHOLD}")
    ax.set(title=f"{op}: effect size dist", xlabel="Cohen's d", ylabel="# features")
    ax.legend()
plt.tight_layout(); plt.savefig("fingerprint_effect_sizes.png", dpi=150); plt.show()

In [ ]:
# ── 10. RSA — representational geometry comparison ────────────────────────────
# Compute RDM (representational dissimilarity matrix) for each operation
# using a random subsample. Compare RDMs via Spearman correlation.
# High correlation = similar representational geometry.

RSA_N = 200   # subsample per operation for tractability

def compute_rdm(feat_matrix, n=RSA_N, seed=0):
    np.random.seed(seed)
    idx = np.random.choice(len(feat_matrix), min(n, len(feat_matrix)), replace=False)
    sub = feat_matrix[idx]
    # Cosine distance
    norms = norm(sub, axis=1, keepdims=True) + 1e-8
    sub_n = sub / norms
    sim   = sub_n @ sub_n.T
    return 1 - sim   # dissimilarity

rdms = {op: compute_rdm(feats[op]) for op in ["add","mul","sub","ctrl"]}

def rdm_corr(rdm_a, rdm_b):
    """Spearman correlation of upper-triangle entries."""
    n = len(rdm_a)
    idx = np.triu_indices(n, k=1)
    rho, _ = spearmanr(rdm_a[idx], rdm_b[idx])
    return rho

all_ops_ctrl = ["add","mul","sub","ctrl"]
print("RDM correlation matrix (Spearman ρ):")
print(f"{'':6s}", "  ".join(f"{o:>6s}" for o in all_ops_ctrl))
for a in all_ops_ctrl:
    row = [rdm_corr(rdms[a], rdms[b]) for b in all_ops_ctrl]
    print(f"{a:6s}", "  ".join(f"{v:>6.3f}" for v in row))

print("\nHypothesis: math ops correlate more with each other than with ctrl")
print("Strong RSA correlation between add and mul → shared representational geometry")

In [ ]:
# ── 11. Constant-trace: variance vs accuracy per operand range ────────────────
# Within each operation, does trace variance across instances predict accuracy?
# Test across single-digit vs double-digit.

print("Constant-trace metric — variance by operand range:")
for op, op_data, op_feats in [("add",add_data,feats["add"]),("mul",mul_data,feats["mul"]),("sub",sub_data,feats["sub"])]:
    print(f"\n  {op}:")
    active = op_feats.max(axis=0) > 0
    range_var, range_acc = [], []
    for range_name in RANGES:
        idx  = [i for i,r in enumerate(op_data) if r["range"]==range_name]
        f    = op_feats[idx]
        v    = float(f[:, active].var(axis=0).mean())
        acc  = np.mean([op_data[i]["correct"] for i in idx])
        range_var.append(v); range_acc.append(acc)
        print(f"    {range_name:8s}  variance={v:.5f}  accuracy={acc:.1%}")
    rho, p = spearmanr(range_var, range_acc)
    print(f"    ρ(variance, accuracy) = {rho:.3f}  p={p:.3f}")

print("\nHypothesis: single-digit (lower variance) → higher accuracy")
print("If ρ < 0 across operations: constant-trace metric predicts competence")

In [ ]:
# ── 12. PCA ───────────────────────────────────────────────────────────────────
# Subsample for tractability
SUB = 500
sub_feats  = {op: feats[op][np.random.choice(len(feats[op]), min(SUB,len(feats[op])), replace=False)]
              for op in ["add","mul","sub","ctrl"]}
X_pca = np.concatenate(list(sub_feats.values()), axis=0)
labels_op = sum([[op]*len(sub_feats[op]) for op in ["add","mul","sub","ctrl"]], [])

pcs = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X_pca))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
op_colors = {"add":"steelblue", "mul":"darkorange", "sub":"green", "ctrl":"tomato"}

# All operations
ax = axes[0]
for op in ["add","mul","sub","ctrl"]:
    m = np.array(labels_op)==op
    ax.scatter(pcs[m,0], pcs[m,1], c=op_colors[op], alpha=0.3, s=15, label=op)
ax.legend(); ax.set_title("PCA: all operations (shared SAE)")

# Math ops only — do they separate?
ax2 = axes[1]
math_mask = np.array(labels_op) != "ctrl"
for op in ["add","mul","sub"]:
    m = np.array(labels_op)==op
    ax2.scatter(pcs[m,0], pcs[m,1], c=op_colors[op], alpha=0.4, s=15, label=op)
ax2.legend(); ax2.set_title("PCA: math operations only")

plt.tight_layout(); plt.savefig("math_pca.png", dpi=150); plt.show()
print("Key: do math ops form a cluster distinct from ctrl (left)?")
print("     do add/mul/sub separate within the math cluster (right)?")

In [ ]:
# ── 13. Correct vs incorrect — fingerprint deviation ─────────────────────────
# Among correct answers, do instances stay closer to the operation centroid?
# This is the hallucination-detection signal: off-fingerprint = likely wrong.

def cosine(a, b): return float(np.dot(a,b) / (norm(a)*norm(b) + 1e-8))

print("Centroid proximity: correct vs incorrect instances")
for op, op_data, op_feats in [("add",add_data,feats["add"]),("mul",mul_data,feats["mul"]),("sub",sub_data,feats["sub"])]:
    centroid = op_feats.mean(axis=0)
    c_idx = [i for i,r in enumerate(op_data) if r["correct"]]
    w_idx = [i for i,r in enumerate(op_data) if r["correct"] == False]
    if len(c_idx) < 5 or len(w_idx) < 5:
        print(f"  {op}: not enough data (correct={len(c_idx)}, incorrect={len(w_idx)})")
        continue
    sim_c = np.mean([cosine(op_feats[i], centroid) for i in c_idx])
    sim_w = np.mean([cosine(op_feats[i], centroid) for i in w_idx])
    delta = sim_c - sim_w
    print(f"  {op}: correct={sim_c:.4f}  incorrect={sim_w:.4f}  Δ={delta:+.4f}  "
          f"({'✓ on-manifold' if delta > 0 else '✗ no signal'})")

In [ ]:
# ── 14. Summary ───────────────────────────────────────────────────────────────
print("=" * 70)
print("SHARED SAE — MATH OPERATION FINGERPRINT SUMMARY")
print("=" * 70)
print(f"  Model:              {MODEL_NAME}  layer {LAYER}")
print(f"  SAE:                shared  d_sae={D_SAE}  k={SAE_K}  epochs={SAE_EPOCHS}")
print(f"  Training corpus:    {len(train_acts)} activation vectors")
print(f"  Variance explained: {var_expl.item():.2%}")
print(f"  Live features:      {(feat_usage > DEAD_THR).sum().item()}/{D_SAE}")
print()
print("  Fingerprint sizes (d > {:.1f}):  add={:d}  mul={:d}  sub={:d}".format(
    FP_THRESHOLD, int(fp_add.sum()), int(fp_mul.sum()), int(fp_sub.sum())))
print(f"  Shared math core (all 3 ops):  {int(all_ops)}")
print()
print(f"  Hierarchy test (add→mul containment): {c_am:.3f}")
print(f"  Hierarchy test (mul→add containment): {c_ma:.3f}")
print(f"  Verdict: {verdict}")
print()
print("NEXT STEPS")
if (feat_usage > DEAD_THR).sum().item() < D_SAE * 0.3:
    print("  ⚠ Many dead features — try smaller k, more epochs, or lower AUX_W")
if var_expl.item() < 0.5:
    print("  ⚠ Low variance explained — try more epochs or larger d_sae")
if verdict == "SUPPORTED":
    print("  → Extend to deeper hierarchy: single-digit add → double-digit add → mul")
    print("  → Test cross-model transfer: do fingerprints align across model sizes?")
    print("  → Build the null-operation detector from the shared math core")
else:
    print("  → Try FP_THRESHOLD=0.3, or increase N_PER_OP")
    print("  → Check if model actually does both operations (accuracy cell above)")
print("=" * 70)